# 09. 远程连接

AGDR VLA 入门教程 · 第 2 章 · 第 2 周

- **学习目标**：能用 SSH 和密钥登录，能用 `scp` / `rsync` 传文件，能用 `tmux` 让任务在断线后继续运行
- **前置知识**：[05-用户与权限](./05-用户与权限.ipynb)、[08-软件安装与压缩包](./08-软件安装与压缩包.ipynb)
- **预计学习时长**：1.5 小时

> 这一节几乎都是交互式操作，**全部在终端里完成**。没有服务器也没关系，SSH 可以连接自己这台机器（`localhost`）来练习，步骤和连接真实服务器完全相同。

## SSH 是什么

SSH 让你在本地终端里安全地操作另一台机器，通信全程加密。训练服务器、机器人上的工控机，通常都这样访问。

```text
你的电脑（SSH 客户端） ──加密连接──▶ 远程机器（SSH 服务端，默认端口 22）
```

基本用法：

```bash
ssh 用户名@地址             # 例如 ssh shiro@192.168.1.100
ssh -p 2222 用户名@地址     # 服务端不用默认端口时
exit                        # 退出（或按 Ctrl + D）
```

> **登录后要看提示符**：确认现在是在哪台机器上，再运行命令。

## 练习准备：让本机可以被连接

Ubuntu 桌面版默认只有 SSH 客户端。在终端里安装服务端：

```bash
sudo apt install openssh-server
systemctl status ssh
```

看到 `active (running)` 或 `active (listening)` 说明服务在运行，按 `q` 退出。如果显示 `inactive`，运行 `sudo systemctl start ssh`。

然后连接自己，并输入**本机用户的密码**：

```bash
ssh localhost
```

第一次连接时会看到类似的提示：

```text
The authenticity of host 'localhost (127.0.0.1)' can't be established.
ED25519 key fingerprint is SHA256:xxxx
Are you sure you want to continue connecting (yes/no/[fingerprint])?
```

SSH 在问：「你确认这台机器是你要连的吗？」输入 `yes` 后，它的指纹会存入 `~/.ssh/known_hosts`。

> 如果以后看到 **REMOTE HOST IDENTIFICATION HAS CHANGED**，不要直接忽略。可能是服务器重装了，也可能有人在中间拦截。先询问服务器管理员。

运行 `exit` 回到原来的会话。

## 密钥登录

每次输密码既麻烦又不够安全，实际工作中更多使用**密钥对**：

| 文件 | 作用 | 能否外传 |
| --- | --- | --- |
| `~/.ssh/id_ed25519.pub`（公钥） | 放到远程机器上，相当于锁 | 可以 |
| `~/.ssh/id_ed25519`（私钥） | 留在自己电脑上，相当于钥匙 | **绝对不行** |

```bash
# 1. 生成密钥对（第 1 章配过 GitHub 的话，这是另一个系统，需要重新生成）
ssh-keygen -t ed25519 -C "你的邮箱"

# 2. 把公钥装到远程机器，写入对方的 ~/.ssh/authorized_keys
ssh-copy-id localhost

# 3. 再次登录，如果设置了密钥口令，这次输入的是口令；否则直接登录
ssh localhost
```

生成时会询问保存位置和口令（passphrase），直接回车使用默认位置即可。口令用于保护私钥，建议设置。

**权限不对时，SSH 会拒绝使用密钥**：

```bash
chmod 700 ~/.ssh
chmod 600 ~/.ssh/id_ed25519
chmod 600 ~/.ssh/authorized_keys
```

这正是第 05 节讲过的 `700` 和 `600`。

## ~/.ssh/config：给服务器起个短名字

用 nano 编辑 `~/.ssh/config`，添加：

```text
Host local
    HostName localhost
    User 你的用户名

Host lab
    HostName 192.168.1.100
    User shiro
    Port 2222
```

之后只要运行 `ssh local`，`scp`、`rsync` 和 VS Code 的 Remote-SSH 扩展也都能使用这些名字。

> `Host lab` 只是格式示例，地址并不真实。

## 传文件：scp 和 rsync

```bash
scp a.txt 用户名@地址:/home/用户名/     # 本地 → 远程
scp 用户名@地址:/path/b.txt ./          # 远程 → 本地
scp -r data/ 用户名@地址:~/             # 复制目录
```

**rsync** 只传有变化的部分，中断后重新运行也不用从头开始，同步大数据集时比 `scp` 更合适：

```bash
rsync -avP data/ 用户名@地址:~/data/
```

| 选项 | 作用 |
| --- | --- |
| `-a` | 归档模式：递归，并保留权限和时间 |
| `-v` | 显示详情 |
| `-P` | 显示进度，并支持断点续传 |
| `-n` | 试运行：只显示会做什么，不真正执行 |
| `--delete` | 删除目标中多余的文件（危险，先配合 `-n` 确认） |

**结尾的 `/` 有区别**：`data/` 表示传目录**里的内容**，`data` 表示传目录**本身**。

可以在本机试一下这个差别：

In [ ]:
%%bash
rm -rf /tmp/linux-demo && mkdir -p /tmp/linux-demo/src && cd /tmp/linux-demo
touch src/a.txt src/b.txt
rsync -a src/ with_slash/
rsync -a src no_slash/
find with_slash no_slash -type f

## tmux：断线不中断任务

通过 SSH 跑训练时，如果网络断开或合上电脑，前台任务通常也会被结束。**tmux** 把终端会话留在远程机器上：你断开了，任务还在继续。

```text
你的电脑 ──SSH──▶ 服务器上的 tmux 会话 ──▶ python train.py
       （断开也没关系）   （一直运行）
```

| 操作 | 命令 / 按键 |
| --- | --- |
| 新建会话 | `tmux new -s train` |
| 暂时离开（detach） | 先按 `Ctrl + B`，松开，再按 `d` |
| 列出会话 | `tmux ls` |
| 回到会话 | `tmux attach -t train` |
| 结束会话 | 在会话里运行 `exit` |
| 新建窗口 / 切换窗口 | `Ctrl + B`，`c` / `Ctrl + B`，`n` |
| 左右分屏 / 上下分屏 | `Ctrl + B`，`%` / `Ctrl + B`，`"` |
| 滚动查看历史 | `Ctrl + B`，`[`，按 `q` 退出 |

注意所有快捷键都要**先按 `Ctrl + B`，松开，再按下一个键**。

### 完整练习

请按顺序亲手做一遍，这也是作业三要记录的内容：

```bash
ssh localhost                 # 1. 模拟登录服务器
tmux new -s train             # 2. 在「服务器」上新建会话
for i in $(seq 1 600); do echo "step $i"; sleep 1; done   # 3. 模拟一个长任务
```

```text
4. 先按 Ctrl + B，再按 d，离开会话，任务继续运行
5. 运行 exit，断开 SSH，也可以把整个终端窗口关掉
```

```bash
ssh localhost                 # 6. 重新登录
tmux ls                       # 7. 会话还在
tmux attach -t train          # 8. 回到会话，step 的数字一直在增长
```

```text
9. 按 Ctrl + C 结束任务，运行 exit 关闭会话
```

**看到第 8 步时 step 数字变大了**，就说明你已经掌握了跑长任务最重要的技能。

> tmux 只能防止断线，**机器重启或关机后，会话也没了**。

## 端口转发：访问远程的 Jupyter

服务器上运行的网页工具（如 Jupyter、TensorBoard）只监听服务器本地时，可以把它的端口「搬」到你的电脑上：

```bash
ssh -L 8888:localhost:8888 用户名@地址
```

连接后，在本地浏览器打开 `http://localhost:8888`，访问到的就是服务器上的 8888 端口。

## 动手试试

在终端里完成，不计分：

1. 用 `scp` 把一个文件从 `/tmp` 复制到 `localhost` 的主目录，确认后删除副本
2. 在 `~/.ssh/config` 里添加 `Host local`，然后用 `ssh local` 登录
3. 说出：公钥和私钥哪个放到服务器？如果 `~/.ssh/id_ed25519` 的权限是 `644`，会发生什么？

<details>
<summary>参考答案</summary>

```bash
echo hi > /tmp/hello.txt
scp /tmp/hello.txt localhost:~/
ls ~/hello.txt
rm ~/hello.txt
ssh local
```

第 3 题：**公钥**放到服务器，私钥留在本地。私钥权限过宽时，SSH 会警告 `UNPROTECTED PRIVATE KEY FILE` 并拒绝使用它，需要用 `chmod 600` 修复。

</details>

## 小结

| 你应该能 | 相关内容 |
| --- | --- |
| 用 SSH 登录，并理解首次连接的指纹提示 | SSH |
| 配置密钥登录，说出公钥和私钥的区别 | 密钥登录 |
| 用 `scp`、`rsync` 传文件，注意结尾的 `/` | 传文件 |
| 用 `tmux` 让任务在断线后继续运行 | tmux |

所有知识都学完了。下一节：[10-综合任务](./10-综合任务.ipynb)